# Parakeet CTC 0.6B tiếng Việt - VIVOS decoder-only

Notebook này chỉ điều phối run. Logic đã tách sang `ASR_local/src/asr_local`:

- `data/vivos.py`: chuẩn bị manifest VIVOS.
- `model/parakeet.py`: cấu hình model/run và tải `.nemo`.
- `train/parakeet_decoder.py`: baseline, train decoder-only, eval, error analysis.
- `common/metrics.py`: normalize/WER/error rows.

## 0. Bootstrap package local + repo main

In [ ]:
from pathlib import Path
import sys


def find_local_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "asr_local").is_dir():
            return candidate
    raise RuntimeError("Không tìm thấy ASR_local/src/asr_local")


LOCAL_ROOT = find_local_root()
LOCAL_SRC = LOCAL_ROOT / "src"
if str(LOCAL_SRC) not in sys.path:
    sys.path.insert(0, str(LOCAL_SRC))

from asr_local.paths import bootstrap

LOCAL_ROOT, MAIN_ROOT = bootstrap(LOCAL_ROOT)
print("LOCAL_ROOT =", LOCAL_ROOT)
print("MAIN_ROOT  =", MAIN_ROOT)

## 1. Cấu hình

In [ ]:
from asr_local.paths import notebook_work_root
from asr_local.model.parakeet import ParakeetConfig, download_parakeet, resolve_device, run_paths

config = ParakeetConfig(
    max_epochs=10,
    train_batch=1,
    accumulate_grad_batches=32,
    eval_batch=8,
    lr=1e-5,
)

WORK_ROOT = notebook_work_root(MAIN_ROOT, "parakeet")
DATA_DIR = WORK_ROOT / "data" / config.run_id
paths = run_paths(WORK_ROOT, config)
device = resolve_device()

print("Thiết bị =", device)
print("Thư mục data =", DATA_DIR)
print("Thư mục run =", paths["run_dir"])

## 2. Chuẩn bị manifest VIVOS

In [ ]:
from asr_local.data.vivos import prepare_vivos_manifests

manifests = prepare_vivos_manifests(
    DATA_DIR,
    train_n=config.train_n,
    val_n=config.val_n,
    test_n=config.test_n,
)

manifests

## 3. Tải pretrained Parakeet `.nemo`

In [ ]:
model_path = download_parakeet(config)
model_path

## 4. Đo WER baseline

In [ ]:
from asr_local.train.parakeet_decoder import baseline_eval

baseline = baseline_eval(model_path, manifests, config, device)
baseline

## 5. Fine-tune decoder-only

In [ ]:
from asr_local.train.parakeet_decoder import train_decoder_only

train_summary = train_decoder_only(
    model_path=model_path,
    manifests=manifests,
    config=config,
    device=device,
    run_dir=paths["run_dir"],
    output_model=paths["finetuned_model"],
)

train_summary

## 6. Đánh giá model đã fine-tune

In [ ]:
from asr_local.train.parakeet_decoder import evaluate_finetuned

results = evaluate_finetuned(
    model_path=paths["finetuned_model"],
    manifests=manifests,
    config=config,
    baseline=baseline,
    train_summary=train_summary,
    device=device,
    results_json=paths["results_json"],
)

results

## 7. Phân tích lỗi

In [ ]:
from asr_local.train.parakeet_decoder import write_error_analysis

errors = write_error_analysis(
    model_path=paths["finetuned_model"],
    manifests=manifests,
    config=config,
    device=device,
    out_csv=paths["error_csv"],
)

errors.head(20)

## 8. Artifact

In [ ]:
from asr_local.train.parakeet_decoder import list_artifacts

list_artifacts(paths["run_dir"])